# One-time setup on HiPerGator (run in Jupyter)

Run this **once**, with the default **Python 3** kernel, top to bottom. It creates the training environment under your `pha` folder on `/blue` and registers a Jupyter kernel named **PHA training**.

After that, reload the browser tab, open `../train_lora.ipynb`, and choose *Kernel → Change kernel → PHA training*.

Expected layout: `/blue/<group>/<gatorlink>/pha/training/hpc/setup_env.ipynb` (this file).

In [ ]:
import os
from pathlib import Path

# The pha folder is two levels above this notebook (pha/training/hpc/). Override if needed:
P = Path.cwd().resolve().parents[1]
# P = Path("/blue/<group>/<gatorlink>/pha")

os.environ["P"] = str(P)
os.environ["UV_CACHE_DIR"] = str(P / "uv-cache")  # keep downloads off the small home quota
os.environ["PATH"] = f"{Path.home() / '.local' / 'bin'}:{os.environ['PATH']}"
print("pha folder:", P)
assert (P / "training" / "requirements-train.txt").is_file(), (
    "Old or missing training/ copy: re-copy it"
)

## 1. Check files and GPU

In [ ]:
%%bash
ls $P/training
echo
echo "Slurm job:          ${SLURM_JOB_ID:-not in a Slurm job}"
[ -n "$SLURM_JOB_ID" ] && scontrol show job "$SLURM_JOB_ID" | grep -oE "Partition=[^ ]+|AllocTRES=[^ ]+"
echo "CUDA_VISIBLE_DEVICES: ${CUDA_VISIBLE_DEVICES:-<unset> (no GPU allocated)}"
echo
nvidia-smi --query-gpu=name,memory.total --format=csv \
  || echo "NO GPU in this session: restart the Jupyter app with gres gpu:l4:1 (partition hpg-turin)"


## 2. Install uv (Python package tool), if missing

In [ ]:
%%bash
command -v uv || curl -LsSf https://astral.sh/uv/install.sh | sh
uv --version

## 3. Create the Python 3.11 env on /blue

In [ ]:
%%bash
uv venv --python 3.11 $P/venv

## 4. Install torch + pinned training libraries (5–10 min)

In [ ]:
%%bash
VIRTUAL_ENV=$P/venv uv pip install torch -r $P/training/requirements-train.txt

## 5. Verify: expect a CUDA version (12.x) and `True`

In [ ]:
%%bash
$P/venv/bin/python -c "import torch, transformers, peft, trl; print('torch', torch.__version__, '| CUDA', torch.version.cuda, '| GPU available', torch.cuda.is_available(), '| transformers', transformers.__version__, '| trl', trl.__version__)"

## 6. Register the **PHA training** kernel (paths point to /blue)

In [ ]:
%%bash
mkdir -p $P/hf-cache $P/runs
$P/venv/bin/python -m ipykernel install --user --name pha-train --display-name "PHA training" \
  --env HF_HOME $P/hf-cache \
  --env PHA_WORK $P/training \
  --env PHA_OUTPUT $P/runs \
  --env LLAMA_CPP_DIR $P/llama.cpp-b11347 \
  --env PATH "$P/venv/bin:$PATH"

## Done

1. Reload this browser tab so Jupyter sees the new kernel.
2. Open `pha/training/train_lora.ipynb`.
3. *Kernel → Change kernel → **PHA training***.
4. Run all cells with `SMOKE_TEST = True` first.